# Notebook de test des modules 

## Chargement des données

In [5]:
import os
import sys

sys.path.append(os.path.abspath('./src'))

from data_loader import load_mtedx_data

try:
    BASE_DIR = os.path.abspath(__file__)
except NameError:
    BASE_DIR = os.path.dirname(".")

DATA_DIR = os.path.join(os.path.abspath(BASE_DIR), 'data')

print(f"Chargement des données depuis {DATA_DIR}...")

datasets = load_mtedx_data(DATA_DIR, pairs=['fr-en','fr-es'])

print("\n---- Résultats du chargement ----")
print("Paires de langues chargées :", datasets.keys())

if 'fr-en' in datasets:
    print("\nSplits disponibles pour fr-en :", datasets['fr-en'].keys())
    print(f"Nombre de segments audio (train): {len(datasets['fr-en']['train'])}")

    print("\nExemple de segment audio (segment 0) :")
    exemple = datasets['fr-en']['train'][0]
    for key, value in exemple.items():
        print(f"{key}: {value[:100]}...")

Chargement des données depuis /home/dylan/COURS/M2_COURS/deep_learning/projet_deeplearning/data...
fr-en - train chargé : 30171 phrases.
fr-en - valid chargé : 1036 phrases.
fr-en - test chargé : 1059 phrases.
fr-es - train chargé : 20826 phrases.
fr-es - valid chargé : 1036 phrases.
fr-es - test chargé : 1059 phrases.

---- Résultats du chargement ----
Paires de langues chargées : dict_keys(['fr-en', 'fr-es'])

Splits disponibles pour fr-en : dict_keys(['train', 'valid', 'test'])
Nombre de segments audio (train): 30171

Exemple de segment audio (segment 0) :
src_text: Je m'appelle Julien Le Breton et, comme mon nom ne l'indique pas, je suis né et j'habite en Nouvelle...
tgt_text: I'm Julien Le Breton and as my name doesn't suggest, I was born and live in New Caledonia....


## Transcription sur un fichier audio

### Transcription

In [6]:
import glob
os.environ["HF_HUB_HTTP_TIMEOUT"] = "120"

from asr_pipeline import AudioTranscriber

OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

dir_wav_test = os.path.join(DATA_DIR, 'fr-fr/data/test/wav')
audio_files = glob.glob(os.path.join(dir_wav_test, '*.flac'))

if not audio_files:
    print(f"Aucun fichier audio trouvé dans {dir_wav_test}")
else:
    test_file = audio_files[0]
    print(f"Fichier audio de test sélectionné : {test_file}")

    transcriber = AudioTranscriber(model_name='openai/whisper-small')
    out_srt_file = os.path.join(OUTPUT_DIR, 'srt/test_output.srt')
    full_text = transcriber.transcribe_and_save(test_file, out_srt_file)

    print("\n--- Aperçu du résultat de la transcription ---")
    print(full_text[:500] + "..." if len(full_text) > 500 else full_text)

Fichier audio de test sélectionné : /home/dylan/COURS/M2_COURS/deep_learning/projet_deeplearning/data/fr-fr/data/test/wav/16MWHwqz0mQ.flac
Chargement du modèle ASR 'openai/whisper-small' sur : cuda:0...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Transcription de '/home/dylan/COURS/M2_COURS/deep_learning/projet_deeplearning/data/fr-fr/data/test/wav/16MWHwqz0mQ.flac' en cours...
Terminé ! Fichier SRT sauvegardé sous : output/srt/test_output.srt

--- Aperçu du résultat de la transcription ---
... Applaudissements Mon quotidien est fait de sensations que la plupart d'entre vous ignorent. Pendant longtemps, j'en ai eu honte. Je me souviens très bien des circonstances dans lesquelles j'ai découvert cette différence. J'avais 4 ans, et j'ai voulu exprimer à ma mère la joie que j'ai ressenti en regardant le soleil à travers une feuille. Et je lui ai parlé non pas de la feuille en tant que telle, mais des couleurs et des sensations qui étaient causées par ce que j'avais ressenti en voyant l...


### Evaluation des performances de la transcription

In [ ]:
from tqdm.notebook import tqdm
from metrics import evaluate_asr

print("\n--- Evaluation de la transcription ---")


In [3]:
from nmt_pipeline import SubtitleTranslator

translate_en = SubtitleTranslator(model_name='Helsinki-NLP/opus-mt-fr-en')

test_sentence = "Bonjour, comment ça va ? Tu savais que l'intelligence artificielle est fascinante ?"
print("\nTest rapide")
print(f"Phrase d'entrée FR : {test_sentence}")
print(f"Traduction EN : {translate_en.translate_text(test_sentence)}")

Chargement du modèle NMT 'Helsinki-NLP/opus-mt-fr-en' sur : cuda:0...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


Test rapide
Phrase d'entrée FR : Bonjour, comment ça va ? Tu savais que l'intelligence artificielle est fascinante ?
Traduction EN : Did you know artificial intelligence is fascinating?


In [4]:
srt_source = os.path.join(OUTPUT_DIR, 'srt/test_output.srt')
srt_target = os.path.join(OUTPUT_DIR, 'srt/test_output_en.srt')

if os.path.exists(srt_source):
    print(f"\nTraduction du fichier SRT {srt_source} vers {srt_target}...")
    translate_en.translate_srt(srt_source, srt_target)
    print("Traduction terminée.")
    print("\n--- Aperçu du contenu du fichier SRT traduit ---")
    with open(srt_target, 'r', encoding='utf-8') as f:
        content = f.read()
        print(content[:500] + "..." if len(content) > 500 else content)
else:
    print(f"\nFichier SRT source {srt_source} introuvable. Veuillez vérifier que la transcription a été effectuée correctement.")


Traduction du fichier SRT output/srt/test_output.srt vers output/srt/test_output_en.srt...
Traduction du fichier 'output/srt/test_output.srt' en cours...
✅ Fichier SRT traduit sauvegardé sous : output/srt/test_output_en.srt
Traduction terminée.

--- Aperçu du contenu du fichier SRT traduit ---
1
00:00:00,000 --> 00:01:06,000
... Applause My daily life is made of sensations that most of you do not know. For a long time, I was ashamed of it. I remember very well the circumstances in which I discovered this difference. I was 4 years old, and I wanted to express to my mother the joy that I felt by looking at the sun through a leaf. And I told her not of the leaf as such, but of the colors and sensations that were caused by what I had felt when I saw the leaf. These yellow and green lines,...
